In [10]:
import pandas as pd
import librosa
import numpy as np
import os

from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier


In [4]:
df_prueba = pd.read_excel('dataset_physionet2016.xlsx')
ciclos_por_archivo = df_prueba.groupby('archivo').size()

# separar el prefijo de carpeta para ver el patron por fuente
ciclos_por_archivo_df = ciclos_por_archivo.reset_index()
ciclos_por_archivo_df.columns = ['archivo', 'ciclos']
ciclos_por_archivo_df['carpeta'] = ciclos_por_archivo_df['archivo'].str.split('_').str[0]

print(ciclos_por_archivo_df.groupby('carpeta')['ciclos'].describe())

             count       mean       std  min  25%   50%   75%   max
carpeta                                                            
training-a   312.0   9.679487  4.123476  2.0  7.0   9.0  11.0  38.0
training-b   446.0   3.986547  1.694618  1.0  3.0   4.0   5.0  10.0
training-c    20.0  10.900000  5.784735  3.0  7.0  10.0  13.0  25.0
training-d    49.0   5.408163  3.201031  2.0  3.0   5.0   7.0  15.0
training-e  1896.0   9.845992  6.674372  2.0  5.0   8.0  12.0  82.0
training-f   102.0  11.588235  5.017500  5.0  8.0  10.5  15.0  28.0


In [5]:

# Definir la ruta base proporcionada
BASE_DIR = r"C:\Users\emigo\OneDrive\Documentos\Servicio Social\classification-of-heart-sound-recordings\classification-of-heart-sound-recordings-the-physionet-computing-in-cardiology-challenge-2016-1.0.0"

archivos_prueba = [
    ('training-b', 'b0001'), 
    ('training-b', 'b0004'), 
    ('training-d', 'd0003'), 
    ('training-d', 'd0016')
]

# revisa la duracion real de un par de archivos de cada carpeta con pocos ciclos
for carpeta, archivo in archivos_prueba:
    # Se recomienda usar os.path.join para evitar errores de sintaxis con las barras en Windows
    ruta = os.path.join(BASE_DIR, carpeta, f"{archivo}.wav")
    
    try:
        x, fs = librosa.load(ruta, sr=None)
        print(f"{carpeta}/{archivo}: duracion={len(x)/fs:.1f}s, fs={fs}")
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo en {ruta}")

training-b/b0001: duracion=8.0s, fs=2000
training-b/b0004: duracion=8.0s, fs=2000
training-d/d0003: duracion=11.0s, fs=2000
training-d/d0016: duracion=11.0s, fs=2000


In [6]:
df_prueba = pd.read_excel('dataset_physionet2016.xlsx')
resumen = df_prueba.groupby('archivo').agg(ciclos=('MFCC_1','size'), duracion=('duracion_s','first'))
resumen['ciclos_por_segundo'] = resumen['ciclos'] / resumen['duracion']
resumen['carpeta'] = resumen.index.str.split('_').str[0]

print(resumen.groupby('carpeta')['ciclos_por_segundo'].describe())

             count      mean       std       min       25%       50%  \
carpeta                                                                
training-a   312.0  0.303030  0.118449  0.154083  0.223839  0.278707   
training-b   446.0  0.499459  0.211907  0.157978  0.375000  0.500000   
training-c    20.0  0.306969  0.184383  0.157587  0.177844  0.259805   
training-d    49.0  0.378745  0.183946  0.166251  0.239234  0.318471   
training-e  1896.0  0.442328  0.201768  0.151362  0.283386  0.403723   
training-f   102.0  0.352325  0.152162  0.152532  0.234067  0.324654   

                 75%       max  
carpeta                         
training-a  0.359071  1.059086  
training-b  0.625000  1.250000  
training-c  0.349595  0.932642  
training-d  0.456053  0.948992  
training-e  0.567763  1.145311  
training-f  0.427180  0.920145  


In [7]:
df_circor = pd.read_excel('dataset_circor.xlsx')   # el v1, NO el v2 -- mismo pipeline que PhysioNet2016
df_circor['fuente'] = 'CirCor'

df_physionet = pd.read_excel('dataset_physionet2016.xlsx')

feature_cols_combo = [f"MFCC_{i+1}" for i in range(13)] + ["RMS"]
columnas_comunes = feature_cols_combo + ['Etiqueta', 'archivo', 'paciente_id', 'fuente']

df_combo = pd.concat([df_circor[columnas_comunes], df_physionet[columnas_comunes]], ignore_index=True)

print(df_combo.groupby('fuente').agg(filas=('Etiqueta','size'), sujetos=('paciente_id','nunique')))
print()
print(df_combo.groupby(['fuente','Etiqueta']).size())

               filas  sujetos
fuente                       
CirCor         11153      586
PhysioNet2016  25131     2825

fuente         Etiqueta
CirCor         0            8649
               2            2504
PhysioNet2016  0           21136
               2            3995
dtype: int64


In [11]:
X_combo = df_combo[feature_cols_combo].values
y_fuente = (df_combo['fuente'] == 'PhysioNet2016').astype(int).values
grupos_combo = df_combo['paciente_id'].astype(str).values

sgkf_auditoria = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
pipe_auditoria = Pipeline([('escalador', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=100, random_state=0))])

scores_auditoria = cross_val_score(pipe_auditoria, X_combo, y_fuente, cv=sgkf_auditoria, groups=grupos_combo, scoring='accuracy')
print(f"Exactitud adivinando la FUENTE (no la etiqueta real): {scores_auditoria.mean():.3f} +/- {scores_auditoria.std():.3f}")
print("(si esto sale muy alto, arriba de 0.90-0.95, es señal de alarma -- el modelo real podria estar usando el mismo atajo)")

Exactitud adivinando la FUENTE (no la etiqueta real): 0.998 +/- 0.001
(si esto sale muy alto, arriba de 0.90-0.95, es señal de alarma -- el modelo real podria estar usando el mismo atajo)
